In [1]:
# ─────────────────────────────────────────
# ID3 — Arbre de décision (Iterative Dichotomiser 3)
# ─────────────────────────────────────────
import math
from collections import Counter

# Données : [Météo, Vent, Jouer?]
dataset = [
    ["Soleil","Faible","Oui"], ["Soleil","Fort","Non"],
    ["Nuage","Faible","Oui"], ["Pluie","Faible","Oui"],
    ["Pluie","Fort","Non"],   ["Nuage","Fort","Oui"],
    ["Soleil","Fort","Non"],  ["Pluie","Faible","Oui"],
]
features = ["Météo", "Vent"]  # Noms des colonnes (sans la cible)

# Calcul de l'entropie d'une liste d'étiquettes
def entropie(labels):
    n = len(labels)
    if n == 0: return 0
    counts = Counter(labels)
    return -sum((c/n) * math.log2(c/n) for c in counts.values() if c)

# Gain d'information d'un attribut (colonne col_idx)
def gain(data, col_idx):
    labels = [r[-1] for r in data]
    e_base = entropie(labels)                      # Entropie avant le split
    valeurs = set(r[col_idx] for r in data)
    e_split = sum(
        (len(sub := [r for r in data if r[col_idx] == v]) / len(data))
        * entropie([r[-1] for r in sub])
        for v in valeurs
    )
    return e_base - e_split                        # Gain = réduction d'entropie

# Construction récursive de l'arbre ID3
def id3(data, feats):
    labels = [r[-1] for r in data]
    if len(set(labels)) == 1: return labels[0]     # Feuille pure
    if not feats: return Counter(labels).most_common(1)[0][0]  # Majorité
    # Choisir l'attribut avec le plus grand gain
    best = max(feats, key=lambda f: gain(data, features.index(f)))
    col  = features.index(best)
    tree = {best: {}}
    for val in set(r[col] for r in data):
        sous = [r for r in data if r[col] == val]
        tree[best][val] = id3(sous, [f for f in feats if f != best])
    return tree

# Construction et affichage
arbre = id3(dataset, features)
print("Arbre ID3 :", arbre)
# → {'Météo': {'Nuage': 'Oui', 'Pluie': 'Oui', 'Soleil': 'Non'}}


Arbre ID3 : {'Vent': {'Fort': {'Météo': {'Nuage': 'Oui', 'Pluie': 'Non', 'Soleil': 'Non'}}, 'Faible': 'Oui'}}
